### Q1. Embedding a query

In [1]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

2026-07-02 07:34:54.244466623 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
print(v[0])

-0.02058203437252893


### Q2. Cosine Similarity

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

In [5]:
doc_vector = embedder.encode(doc["content"])

In [6]:
similarity = v.dot(doc_vector)

print(similarity)

0.36107027225589694


### Q.3 Chunking

In [7]:
from gitsource import chunk_documents
import numpy as np
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [8]:
X = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [9]:
scores = X.dot(v)

In [10]:
idx = np.argmax(scores)

chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

### Q.4 Vector Search

In [22]:
from minsearch import VectorSearch

vs = VectorSearch(
    keyword_fields=["filename"],
)

vs.fit(
    vectors=X,
    payload=chunks
)

In [23]:
query = "What metric do we use to evaluate a search engine?"

query_vector = embedder.encode(query)

In [24]:
results = vs.search(
    query_vector=query_vector,
    num_results=5
)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'